# MESA Basics


## Setup

In [ ]:
#bash
!cp MESA-Web_Job_04012666280/trimmed_history.data MESA-Web_Job_04012666280/history.data
!curl -fsSL https://pixi.sh/install.sh | sh
# Windows Version: !powershell -ExecutionPolicy Bypass -c "irm -useb https://pixi.sh/install.ps1 | iex"
!pixi install
!pixi run register-kernel

In [ ]:
import matplotlib.pyplot as plt
import mesa_reader as mr
import numpy as np
import pandas as pd
import math

## Load History and Profile Data

In [ ]:
# Load your history file. Note directory location specified.
h = mr.MesaData('./MESA-Web_Job_04012666280/trimmed_history.data')
# https://www.pas.rochester.edu/~emamajek/EEM_dwarf_UBVIJHK_colors_Teff.txt
zams = pd.read_csv('mamajek_table.txt', sep='\\s+')

# Load the entire directory
l = mr.MesaLogDir('./MESA-Web_Job_04012666280')

# Select a specific profile:
# Use l.profile_data() for the very last model,
# or l.profile_data(model_number=100) for a specific step.
p = l.profile_data(model_number=1000)

In [ ]:
print(f'history.profile columns: {h.bulk_names}\n\n ZAMS columns: {zams.columns}\n\n specfiic profile columns {p.bulk_names}')

## X by star_age history plots

In [ ]:
bulk_names = list(h.bulk_names)
bulk_names.remove('star_age')

COLS = 4
SECTION_SIZE = 20
sections = [bulk_names[i:i+SECTION_SIZE] for i in range(0, len(bulk_names), SECTION_SIZE)]

for sec_num, section in enumerate(sections):
    rows = math.ceil(len(section) / COLS)
    fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 4, rows * 3))
    axes = axes.flatten()

    for i, name in enumerate(section):
        try:
            axes[i].plot(h.star_age, getattr(h, name), lw=1)
            axes[i].set_title(name, fontsize=8)
            axes[i].set_xlabel('Age (log yr)', fontsize=7)
            axes[i].tick_params(labelsize=6)
        except Exception as e:
            axes[i].set_title(f'{name}\n(error)', fontsize=7, color='red')

    for j in range(len(section), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f'History Variables: Section {sec_num + 1}/{len(sections)}', fontsize=10)
    plt.tight_layout()
#    plt.savefig(f'history_section_{sec_num + 1}.png', dpi=150, bbox_inches='tight')
    plt.show()

## X by PROP profile plots

In [ ]:
profiles_of_interest = {
    1: '1',    # Starting Point
    2: '50',   # 
    7: '275',  # Main sequence
    9: '325',  # Main sequence
    11: '362', # Main sequence
    14: '467', # Star_mass starts dropping, log_L starts decreasing, log_R starts decreasing, center_he4 starts dropping fast, center_c12 starts increasing rapidly, center_n14 drops very fast, center_entropy begins to slowly drop, compactness_parameter increases significantly, dynamic_timescale inflection point - starts decreasing, tri_alfa reaches constant at 0, he_core_mass slope increases: He shell ignition, Asymptotic Giant Branch begin (AGB)
    15: '476', # log_LH inflection point from neg to pos slope: Early-AGB to Thermally Pulsing-AGB transition
    17: '545', # center_c12 starts to increase significantly, center_o16 starts to increase significantly, compactness_parameter slope decreases but is still pos: late AGB, C/O core building
    27: '1000',# sharp log_Teff jump, sharp log_L jump, middle of strong center_degeneracy slope increase to 85, compactness_parameter reaching 0, log_LH oscillations before extreme sharp jump, cno pp oscillations, right before dynamic_timescale sharp down and up jump, o_core_mass reaches maximum, he_core_radius increasing rapidly: Onset of core collapse
}

PROP = 'mass'
COLS = 2
SECTION_SIZE = 60

for prof_num, label in profiles_of_interest.items():
    p = l.profile_data(profile_number=prof_num)
    model_num = p.model_number
    age = h.star_age[h.model_number == model_num][0]

    bulk_names = [n for n in p.bulk_names if n != PROP]
    rows = math.ceil(len(bulk_names) / COLS)
    fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 4, rows * 3))
    axes = axes.flatten()

    for i, name in enumerate(bulk_names):
        try:
            axes[i].plot(getattr(p, PROP), getattr(p, name), lw=1)
            axes[i].set_title(name, fontsize=8)
            axes[i].set_xlabel(PROP, fontsize=7)
            axes[i].tick_params(labelsize=6)
        except Exception as e:
            axes[i].set_title(f'{name}\n(error)', fontsize=7, color='red')

    for j in range(len(section), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(
        f'Profile {prof_num} | Model {model_num} | Age: {age:.3e} yr', fontsize=11
    )
    plt.tight_layout(rect=[0, 0, 1, 0.97])
#   fig.savefig(f'profile_{prof_num:02d}_model_{model_num}.png', dpi=150, bbox_inches='tight')
    plt.show()

## X by model_number history plots

In [ ]:
bulk_names = list(h.bulk_names)
bulk_names.remove('model_number')

COLS = 10
SECTION_SIZE = 60
sections = [bulk_names[i:i+SECTION_SIZE] for i in range(0, len(bulk_names), SECTION_SIZE)]

for sec_num, section in enumerate(sections):
    rows = math.ceil(len(section) / COLS)
    fig, axes = plt.subplots(rows, COLS, figsize=(COLS * 4, rows * 3))
    axes = axes.flatten()

    for i, name in enumerate(section):
        try:
            axes[i].plot(h.model_number, getattr(h, name), lw=1)
            axes[i].set_title(name, fontsize=8)
            axes[i].set_xlabel('model_number', fontsize=7)
            axes[i].tick_params(labelsize=6)
            # Profiles of interest
            for prof_num in profiles_of_interest:
                p_temp = l.profile_data(profile_number=prof_num)
                axes[i].axvline(p_temp.model_number, color='red', alpha=0.5, lw=1)
                axes[i].text(p_temp.model_number, axes[i].get_ylim()[1], str(prof_num), fontsize=7, color='red')
        except Exception as e:
            axes[i].set_title(f'{name}\n(error)', fontsize=7, color='red')

    for j in range(len(section), len(axes)):
        axes[j].set_visible(False)

    fig.suptitle(f'History Variables: Section {sec_num + 1}/{len(sections)}', fontsize=10)
    plt.tight_layout()
    plt.savefig(f'model_number_{sec_num + 1}.png', dpi=150, bbox_inches='tight')
    plt.show()

## Filters

In [ ]:
# HR mask for 12M_sun
Amask = (h.log_Teff >= 4) & (h.log_Teff <=4.4) & (h.log_L <= 4.2)
Bmask = (h.log_Teff >= 3.7) & (h.log_Teff <= 4) & (h.log_L <= 4)
mask = Amask + Bmask

# ZAMS
zams = zams[zams['logL'] != '...']
zams = zams[zams['logT'] != '...']
zams['logL'] = zams['logL'].astype(float)
zams['logT'] = zams['logT'].astype(float)

## Hertzsprung-Russell Diagram

In [ ]:
plt.figure(figsize=(16, 12))

# ZAMS reference line
plt.plot(zams['logT'], zams['logL'], 'b--', label='ZAMS (Pecaut & Mamajek 2013)', zorder=1)

# MESA output
plt.plot(h.log_Teff, h.log_L, color='tab:blue', lw=2, label='12M_sun MESA')
plt.plot(h.log_Teff[mask], h.log_L[mask], color='red', label='12M_sun MESA Main Sequence')

short_labels = {
    1: 'Start',
    2: 'Contraction',
    7: 'TAMS',
    9: 'RGB base',
    11: 'TAHB',
    14: 'AGB begin',
    15: 'TP-AGB',
    17: 'Late AGB',
    27: 'Core Collapse',
}

offsets = {
    1:  (6, -15),
    2:  (-30, -25),
    7:  (-50, -15),
    9:  (6, -15),
    11: (-65, 15),
    14: (30, -20),
    15: (30, 0),
    17: (30, 20),
    27: (-70, 20),
}

for prof_num, model_str in profiles_of_interest.items():
    model_num = int(model_str)
    idx = np.where(h.model_number == model_num)[0]
    if len(idx) == 0:
        continue
    idx = idx[0]
    x = h.log_Teff[idx]
    y = h.log_L[idx]
    plt.scatter(x, y, color='red', s=10, zorder=5)
    plt.annotate(
        short_labels[prof_num],
        xy=(x, y),
        xytext=offsets[prof_num],
        textcoords='offset points',
        fontsize=7,
        arrowprops=dict(arrowstyle='-', color='gray', lw=0.5)
    )

plt.gca().invert_xaxis()  # Hotter stars on the left
plt.xlabel(r'$\log_{10}(T_{\rm eff} [ {\rm K}])$')
plt.ylabel(r'$\log_{10}(L / L_{\odot})$')
plt.title('Hertzsprung-Russell Diagram')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.6)

plt.savefig('hr_diagram.png', dpi=150, bbox_inches='tight')
plt.show()

## (Profile) Internal Structure: Log T by log Rho

In [ ]:
plt.figure(figsize=(8, 6))

# Plot logT vs logRho
plt.plot(p.logRho, p.logT, color='black', label='Stellar Interior')

plt.xlabel(r'$\log_{10}(\rho  [\rm{g\,cm}^{-3}])$')
plt.ylabel(r'$\log_{10}(T  [\rm{K}])$')
plt.title(f'Internal Structure (Model {p.model_number})')
plt.grid(True, linestyle=':', alpha=0.7)

plt.legend()
plt.show()

## (History) Nuclear burning and Central temperature history

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 12), sharex=True)

# Mass fractions
axes[0].plot(h.model_number, h.center_h1, label='center_h1', lw=1.5)
axes[0].plot(h.model_number, h.center_he4, label='center_he4', lw=1.5)
axes[0].set_ylabel('Mass Fraction')
axes[0].set_title('Core Composition')
axes[0].legend(fontsize=8)
axes[0].grid(True, linestyle='--', alpha=0.5)

# Nuclear Burning
axes[1].plot(h.model_number, h.cno, label='cno', lw=1.5)
axes[1].plot(h.model_number, h.tri_alfa, label='tri_alfa', lw=1.5)
axes[1].set_ylabel('log Burning Rate')
axes[1].set_title('Nuclear Burning Rates')
axes[1].legend(fontsize=8)
axes[1].grid(True, linestyle='--', alpha=0.5)

# Central Temperature
axes[2].plot(h.model_number, h.log_center_T, color='black', lw=1.5)
axes[2].set_ylabel('log_center_T')
axes[2].set_title('Central Temperature Evolution')
axes[2].grid(True, linestyle='--', alpha=0.5)

# Interesting Model Numbers
for prof_num, model_str in profiles_of_interest.items():
    for ax in axes:
        ax.axvline(int(model_str), color='red', alpha=0.5, lw=1)
        ax.text(int(model_str), ax.get_ylim()[1], str(prof_num), fontsize=7, color='red', ha='center')

axes[2].set_xlabel('Model Number')
fig.suptitle('Evolution', fontsize=12)
plt.tight_layout(rect=[0, 0, 1, 0.96])
plt.savefig('Evolution.png', dpi=150, bbox_inches='tight')
plt.show()

## (History) Log Age by (Mass and Log_center_p) Luminosity

In [ ]:
fig, ax = plt.subplots()
ax.plot(h.star_age, h.log_center_P)
ax.set_xlabel('Age (years)')
ax.set_ylabel('Log(Center P)')
ax.set_title('Log Central Pressure by Age')
plt.savefig('age_by_centerP.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
fig, ax = plt.subplots()
ax.plot(h.star_age, h.star_mass)
ax.set_xlabel('Age (years)')
ax.set_ylabel('mass')
ax.set_title('Mass by Age')
plt.savefig('age_by_mass.png', dpi=150, bbox_inches='tight')
plt.show()

## (History) Model Number by tri_alfa

In [ ]:
fig, ax = plt.subplots()
ax.plot(h.model_number, h.tri_alfa)
ax.set_xlabel('model number')
ax.set_ylabel('tri_alfa')
ax.set_title('Triple-alpha burning rate over time')

# Profiles of interest
for prof_num in profiles_of_interest:
    p_temp = l.profile_data(profile_number=prof_num)
    ax.axvline(p_temp.model_number, color='red', alpha=0.5, lw=1)
    ax.text(p_temp.model_number, ax.get_ylim()[1], str(prof_num), fontsize=7, color='red')

plt.show()

## (History) Model Number by pp

In [ ]:
fig, ax = plt.subplots()
ax.plot(h.model_number, h.pp)
ax.set_xlabel('model number')
ax.set_ylabel('pp')
ax.set_title('pp burning rate over time')

# Profiles of interest
for prof_num in profiles_of_interest:
    p_temp = l.profile_data(profile_number=prof_num)
    ax.axvline(p_temp.model_number, color='red', alpha=0.5, lw=1)
    ax.text(p_temp.model_number, ax.get_ylim()[1], str(prof_num), fontsize=7, color='red')

plt.tight_layout(rect=[0, 0, 1, 0.97])
plt.show()